# Random Forest na Classificação de Presença no ENEM

# Seção 1: Importação do Dataset e Dataset Usado

In [1]:
import pandas as pd
import string

In [ ]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_SEXO','TP_ESTADO_CIVIL', 'TP_COR_RACA', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'TP_ENSINO', 'IN_TREINEIRO', 
           'TP_DEPENDENCIA_ADM_ESC', 'NU_ANO']

df = pd.read_parquet("enem_parquet", columns = colunas)

# Seção 2: Tratamento de Dados e Criação de Features

In [ ]:
def transformar_colunas(df):

    colunas = [
    'Q001','Q002','Q003','Q004','Q006','Q007','Q008','Q009','Q010',
    'Q011','Q012','Q013','Q014','Q015','Q016','Q017','Q018',
    'Q019','Q020','Q021', 'Q022','Q023','Q024','Q025'
    ]

    letras = {l:i for i,l in enumerate(string.ascii_uppercase)}

    for col in colunas:

        df[col] = df[col].map(letras)

        df[col] = df[col].fillna(-1)

    return df

In [ ]:
df = transformar_colunas(df)

In [ ]:
df

In [ ]:
df["IDADE_ALTA"] = (df["TP_FAIXA_ETARIA"] > 5).astype(int)

df["ACESSO_DIGITAL"] = (
    (df["Q024"] > 0) & (df["Q025"] == 1)  
).astype(int)

df['PAIS_SEM_INSTRUCAO'] = (
    (df['Q001'] == 0) & (df['Q002'] == 0)
).astype(int)

df['SEM_DIGITAL'] = (
    (df['Q024'] == 0) & (df['Q025'] == 0)
).astype(int)

df['FAMILIA_GRANDE_POBRE'] = (
    (df['Q005'] >= 4) &   
    (df['Q006'] <= 2)     
).astype(int)

df['SCORE_BENS_RELEVANTES'] = (
    df['Q010'] +   
    df['Q014'] +   
    df['Q021'] +   
    df['Q024'] +  
    df['Q025']     
)

df['TREINEIRO_JOVEM'] = (
    (df['IN_TREINEIRO'] == 1) &
    (df['TP_FAIXA_ETARIA'] <= 2)   
).astype(int)

df['FAMILIA_QUALIFICADA'] = (
    (df['Q003'] >= 4) |  
    (df['Q004'] >= 4)    
).astype(int)

df['SCORE_RISCO'] = (
    (df['Q006'] <= 2).astype(int) * 2 +  
    (df['Q004'] >= 2).astype(int) +       
    (df['IN_TREINEIRO'] == 1).astype(int)* 2 + # treineiro — peso maior
    (df['IDADE_ALTA'] == 1).astype(int)        # idade alta
)

df['MORA_SOZINHO'] = (df['Q005'] == 1).astype(int)

df['PUBLICO_CURSANDO'] = (
    (df['TP_ESCOLA'] == 1) &       # escola pública
    (df['TP_ST_CONCLUSAO'] == 1)   # ainda cursando
).astype(int)

bens = ['Q007','Q008','Q009','Q010','Q011','Q012','Q013','Q014']
df['SEM_BENS'] = (df[bens].max(axis=1) == 0).astype(int)

In [ ]:
df['TP_ENSINO'] = df['TP_ENSINO'].fillna(0)
df['TP_DEPENDENCIA_ADM_ESC'] = df['TP_DEPENDENCIA_ADM_ESC'].fillna(0)

dicionario_genero = {'F': 0, 'M': 1}
df['TP_SEXO'] = df['TP_SEXO'].map(dicionario_genero)

In [ ]:
FALTOU = (
    (df['TP_PRESENCA_CH'] == 0) & 
    (df['TP_PRESENCA_LC'] == 0) & 
    (df['TP_PRESENCA_CN'] == 0) & 
    (df['TP_PRESENCA_MT'] == 0)
)

df['FALTOU'] = FALTOU.astype(int)

In [ ]:
df['MEDIA'] = (df['NU_NOTA_CN'] + df['NU_NOTA_CH'] + df['NU_NOTA_MT']+  df['NU_NOTA_LC'] + df['NU_NOTA_REDACAO']) / 5
df['CLASSE_NOTA'] = df['MEDIA'].apply(lambda x: 0 if x >= 540 else 1 )

In [ ]:
df

In [ ]:
df.to_parquet("dados_treino_nota.parquet")